# Notebook 04 - RAG Explainability and Audit Reports

This notebook turns model outputs from Notebook 03 into readable audit-style reports.

It uses a lightweight Retrieval-Augmented Generation design:

1. Build a small procurement red-flag knowledge base.
2. Retrieve the most relevant guidance for each flagged contract.
3. Generate a strict JSON risk report with a Pydantic schema.
4. Save reports, retrieved context, and a simple evaluation file.

The default report generator is deterministic and does not require an API key. This keeps the final demo reliable in Colab or local Jupyter.

## 0. Setup

In [ ]:
from pathlib import Path
import sys
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('Running in:', 'Google Colab' if IN_COLAB else 'Local Jupyter')

if IN_COLAB:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'scikit-learn', 'pydantic', 'pandas', 'numpy', 'matplotlib', 'joblib'
    ], check=True)

ROOT = Path.cwd()

# Notebook 03 outputs may be in the current folder or in outputs_03.
MODEL_DIR = ROOT
if not (MODEL_DIR / 'model_comparison.csv').exists() and (ROOT / 'outputs_03' / 'model_comparison.csv').exists():
    MODEL_DIR = ROOT / 'outputs_03'

# Notebook 02 feature data may be in the current folder or in outputs_02.
FEATURE_DIR = ROOT
if not (FEATURE_DIR / 'contracts_ie_features.csv').exists() and (ROOT / 'outputs_02' / 'contracts_ie_features.csv').exists():
    FEATURE_DIR = ROOT / 'outputs_02'

OUTPUT_DIR = ROOT / 'outputs_04'
OUTPUT_DIR.mkdir(exist_ok=True)

print('FEATURE_DIR:', FEATURE_DIR)
print('MODEL_DIR  :', MODEL_DIR)
print('OUTPUT_DIR :', OUTPUT_DIR)

In [ ]:
import importlib
import json
import math
import subprocess
import sys
import warnings
from typing import List, Optional

# Keep the notebook plug-and-play in fresh Colab/Jupyter sessions.
def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    try:
        return importlib.import_module(import_name)
    except ImportError:
        print(f'Installing missing package: {package_name}')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package_name], check=True)
        return importlib.import_module(import_name)

np = ensure_package('numpy')
pd = ensure_package('pandas')
plt = ensure_package('matplotlib', 'matplotlib.pyplot')
ensure_package('pydantic')
ensure_package('scikit-learn', 'sklearn')

from pydantic import BaseModel, Field
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
RANDOM_STATE = 42


## 1. Load Model Outputs

In [ ]:
required_files = [
    FEATURE_DIR / 'contracts_ie_features.csv',
    MODEL_DIR / 'model_comparison.csv',
    MODEL_DIR / 'anomaly_scores.csv',
]

missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')

# Load the three files produced by Notebooks 02 and 03.
df = pd.read_csv(FEATURE_DIR / 'contracts_ie_features.csv', low_memory=False)
model_comparison = pd.read_csv(MODEL_DIR / 'model_comparison.csv')
anomaly_scores = pd.read_csv(MODEL_DIR / 'anomaly_scores.csv')

# Clean possible CSV header issues from Colab uploads or Excel round-trips.
for frame in [df, model_comparison, anomaly_scores]:
    frame.columns = frame.columns.astype(str).str.replace('\ufeff', '', regex=False).str.strip()

# Support older files that used id instead of contract_id.
if 'contract_id' not in df.columns and 'id' in df.columns:
    df = df.rename(columns={'id': 'contract_id'})
if 'contract_id' not in anomaly_scores.columns and 'id' in anomaly_scores.columns:
    anomaly_scores = anomaly_scores.rename(columns={'id': 'contract_id'})

print(f'contracts_ie_features.csv : {df.shape}')
print(f'model_comparison.csv      : {model_comparison.shape}')
print(f'anomaly_scores.csv        : {anomaly_scores.shape}')
print()
print('Feature columns preview:')
print(list(df.columns[:20]))

model_comparison


## 2. Build Audit Dataset

In [ ]:
if 'contract_id' not in df.columns:
    raise ValueError(
        'contracts_ie_features.csv must contain contract_id. '
        f'Current columns are: {list(df.columns)}. '
        'Check that you loaded the Notebook 02 output, not model_comparison.csv or another file.'
    )

# If Notebook 02 was run before NER integration, merge NER features here as a fallback.
if 'shared_address_flag' not in df.columns:
    import ast

    ner_candidates = [
        FEATURE_DIR / 'ner_features (full).csv',
        FEATURE_DIR / 'ner_features.csv',
        ROOT / 'ner_features (full).csv',
        ROOT / 'ner_features.csv',
    ]
    ner_path = next((path for path in ner_candidates if path.exists()), None)

    if ner_path is not None:
        print(f'Merging NER features from: {ner_path.name}')
        ner_df = pd.read_csv(ner_path, low_memory=False)
        ner_df.columns = ner_df.columns.astype(str).str.replace('﻿', '', regex=False).str.strip()

        def list_count(value):
            if pd.isna(value):
                return 0
            try:
                parsed = ast.literal_eval(str(value))
                return len(parsed) if isinstance(parsed, list) else 0
            except Exception:
                return 0

        ner_df['shared_address_flag'] = pd.to_numeric(
            ner_df['shared_address_flag'], errors='coerce'
        ).fillna(0).astype(int)
        ner_df['num_extracted_companies'] = ner_df['extracted_companies'].apply(list_count)
        ner_df['num_extracted_locations'] = ner_df['extracted_locations'].apply(list_count)
        ner_df['has_extracted_company'] = (ner_df['num_extracted_companies'] > 0).astype(int)
        ner_df['has_extracted_location'] = (ner_df['num_extracted_locations'] > 0).astype(int)

        df = df.merge(ner_df, on='contract_id', how='left', validate='one_to_one')
    else:
        print('NER feature file not found. Continuing without NER columns.')
        df['shared_address_flag'] = 0
        df['extracted_companies'] = '[]'
        df['extracted_locations'] = '[]'
        df['num_extracted_companies'] = 0
        df['num_extracted_locations'] = 0
        df['has_extracted_company'] = 0
        df['has_extracted_location'] = 0

for col in ['shared_address_flag', 'num_extracted_companies', 'num_extracted_locations',
            'has_extracted_company', 'has_extracted_location']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
for col in ['extracted_companies', 'extracted_locations']:
    if col in df.columns:
        df[col] = df[col].fillna('[]')

label_cols = ['single_bid', 'winner_concentration', 'copy_paste_description', 'shared_address_flag']
missing_labels = [col for col in label_cols if col not in df.columns]
if missing_labels:
    raise ValueError(f'Missing label columns: {missing_labels}')

for col in label_cols + ['short_tender_period']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df['target_suspicious'] = df[label_cols].eq(1).any(axis=1).astype(int)

merge_cols = ['contract_id', 'anomaly_score', 'isolation_forest_flag']
missing_anomaly = [col for col in merge_cols if col not in anomaly_scores.columns]
if missing_anomaly:
    raise ValueError(f'anomaly_scores.csv is missing columns: {missing_anomaly}')

audit_df = df.merge(
    anomaly_scores[merge_cols],
    on='contract_id',
    how='left'
)

audit_df['isolation_forest_flag'] = pd.to_numeric(
    audit_df['isolation_forest_flag'], errors='coerce'
).fillna(0).astype(int)
audit_df['anomaly_score'] = pd.to_numeric(audit_df['anomaly_score'], errors='coerce')

audit_df['text_for_report'] = audit_df['description'].fillna('').astype(str).str.strip()
empty_text = audit_df['text_for_report'].eq('')
audit_df.loc[empty_text, 'text_for_report'] = audit_df.loc[empty_text, 'title'].fillna('').astype(str).str.strip()

print(f'Audit rows: {len(audit_df):,}')
print(audit_df[['target_suspicious', 'shared_address_flag', 'isolation_forest_flag']].mean().rename('rate'))


## 3. Procurement Knowledge Base

In [ ]:
knowledge_chunks = [
    {
        'source': 'EU procurement principle - competition',
        'topic': 'single bidding',
        'text': (
            'Single-bid procedures are a common procurement red flag because they may indicate weak competition, narrow specifications, limited market outreach, or a tender design that discouraged alternative bidders. A single-bid flag is not proof of misconduct, but it should prompt review of the procedure type, publication quality, deadlines, contract specificity, and buyer history.'
        )
    },
    {
        'source': 'EU procurement principle - equal treatment',
        'topic': 'short tender period',
        'text': (
            'A short advertisement or tender period can reduce effective competition because suppliers have less time to prepare bids. Reviewers should check whether the deadline was proportionate to contract complexity, whether urgency was justified, and whether the short period coincides with other red flags such as single bidding or repeated awards.'
        )
    },
    {
        'source': 'Procurement integrity typology - winner concentration',
        'topic': 'winner concentration',
        'text': (
            'Winner concentration describes a pattern where the same buyer repeatedly awards contracts to a small set of suppliers or where one supplier dominates a buyer relationship. It can reflect legitimate specialization, but it can also indicate supplier lock-in, weak competition, or favoritism. It should be interpreted with buyer and market context.'
        )
    },
    {
        'source': 'NLP similarity red flag - copy-paste descriptions',
        'topic': 'copy paste',
        'text': (
            'Near-identical contract descriptions across different buyers can indicate reused templates, centralized procurement, framework agreements, or potentially tailored specifications. High cosine similarity should be treated as an investigative cue rather than a fraud conclusion. Human review should compare contract titles, buyers, CPV category, publication dates, and description details.'
        )
    },
    {
        'source': 'NER entity-linking red flag - shared address or location',
        'topic': 'entity linking',
        'text': (
            'Named Entity Recognition can extract organization and location mentions from tender text. Shared addresses or recurring location mentions across supposedly independent records can be legitimate, but they may also indicate parent-company relationships, shared administration, or shell-company patterns that deserve human review.'
        )
    },
    {
        'source': 'Unsupervised anomaly detection guidance',
        'topic': 'anomaly detection',
        'text': (
            'Embedding-space anomalies are contracts whose language or metadata differ from the broader corpus. Isolation Forest flags unusual records without using fraud labels, so its results should be used to prioritize review rather than to classify misconduct. Stronger concern arises when anomaly flags overlap with interpretable procurement indicators.'
        )
    },
    {
        'source': 'Academic prototype disclaimer',
        'topic': 'responsible use',
        'text': (
            'This system is an academic decision-support prototype. A suspicious label, model score, similarity score, or anomaly flag is not evidence of fraud, collusion, or illegal behavior. Outputs should be framed as risk indicators requiring human review and additional source documentation.'
        )
    },
    {
        'source': 'Model evaluation guidance - imbalanced labels',
        'topic': 'evaluation',
        'text': (
            'Procurement risk labels are imbalanced and partly missing. Precision-recall AUC is more informative than accuracy because reviewers care about how many flagged cases are useful and how many suspicious cases are missed. Proxy labels are useful for modeling, but they should be discussed as noisy indicators.'
        )
    }
]

knowledge_df = pd.DataFrame(knowledge_chunks)
knowledge_df.to_csv(OUTPUT_DIR / 'rag_knowledge_chunks.csv', index=False)
knowledge_df

## 4. Build Retriever

In [ ]:
retriever = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
chunk_matrix = retriever.fit_transform(knowledge_df['text'])

def retrieve_context(query, top_k=3):
    query_vec = retriever.transform([query])
    scores = cosine_similarity(query_vec, chunk_matrix).ravel()
    top_idx = scores.argsort()[-top_k:][::-1]
    results = []
    for idx in top_idx:
        row = knowledge_df.iloc[idx]
        results.append({
            'source': row['source'],
            'topic': row['topic'],
            'text': row['text'],
            'score': float(scores[idx])
        })
    return results

for query in ['single bid no competition', 'copy paste descriptions different buyers', 'anomaly detection unusual contract']:
    top = retrieve_context(query, top_k=1)[0]
    print(f'Query: {query}')
    print(f'  -> {top["source"]} | score={top["score"]:.3f}')
    print(f'     {top["text"][:160]}...\n')

## 5. Strict Report Schema

In [ ]:
class RetrievedContext(BaseModel):
    source: str
    topic: str
    score: float
    text: str

class RiskReport(BaseModel):
    contract_id: str
    title: str
    buyer_name: Optional[str] = None
    risk_score: int = Field(ge=0, le=10)
    risk_level: str
    risk_factors: List[str]
    retrieved_context: List[RetrievedContext]
    explanation: str
    human_review_questions: List[str]
    disclaimer: str

print('Pydantic schema ready.')

## 6. Deterministic RAG Report Generator

In [ ]:
def clean_bool(value):
    try:
        return int(float(value)) == 1
    except Exception:
        return False

def build_risk_factors(row):
    factors = []
    if clean_bool(row.get('single_bid')):
        factors.append('single_bid')
    if clean_bool(row.get('winner_concentration')):
        factors.append('winner_concentration')
    if clean_bool(row.get('copy_paste_description')):
        factors.append('copy_paste_description')
    if clean_bool(row.get('shared_address_flag')):
        factors.append('shared_address_flag')
    if clean_bool(row.get('isolation_forest_flag')):
        factors.append('embedding_anomaly')
    if clean_bool(row.get('short_tender_period')):
        factors.append('short_tender_period_observed_but_not_in_main_label')
    return factors

def score_risk(factors, anomaly_score=None):
    score = 1
    weights = {
        'single_bid': 3,
        'winner_concentration': 2,
        'copy_paste_description': 2,
        'shared_address_flag': 2,
        'embedding_anomaly': 2,
        'short_tender_period_observed_but_not_in_main_label': 1,
    }
    for factor in factors:
        score += weights.get(factor, 1)
    if anomaly_score is not None and not pd.isna(anomaly_score):
        score += 1 if anomaly_score > audit_df['anomaly_score'].quantile(0.95) else 0
    return int(min(score, 10))

def risk_level(score):
    if score >= 7:
        return 'high'
    if score >= 4:
        return 'medium'
    return 'low'

def query_from_row(row, factors):
    return ' '.join([
        str(row.get('title', '')),
        str(row.get('description', ''))[:500],
        ' '.join(factors),
        str(row.get('buyer_name', '')),
    ])

def make_review_questions(factors):
    questions = []
    if 'single_bid' in factors:
        questions.append('Was the tender specification broad enough to allow meaningful competition?')
    if 'winner_concentration' in factors:
        questions.append('Does this buyer repeatedly award similar contracts to the same supplier group?')
    if 'copy_paste_description' in factors:
        questions.append('Are the highly similar descriptions explained by a legitimate shared template or framework agreement?')
    if 'shared_address_flag' in factors:
        questions.append('Do extracted organization/location mentions suggest a legitimate shared site, parent entity, or administrative address?')
    if 'embedding_anomaly' in factors:
        questions.append('What language or metadata makes this notice unusual compared with the corpus?')
    if not questions:
        questions.append('Is there any external evidence that would justify manual review despite low model risk?')
    questions.append('Can the risk signal be verified against the original tender notice and award documentation?')
    return questions

def explain_contract(row, top_k=3):
    factors = build_risk_factors(row)
    score = score_risk(factors, row.get('anomaly_score'))
    contexts = retrieve_context(query_from_row(row, factors), top_k=top_k)

    if factors:
        factor_text = ', '.join(factors).replace('_', ' ')
        explanation = (
            f'This contract receives a {risk_level(score)} risk rating because it matches the following proxy indicators: '
            f'{factor_text}. The retrieved guidance frames these signals as review triggers, not proof of wrongdoing. '
            'A human reviewer should compare the model flags with the original tender documents, buyer history, and market context.'
        )
    else:
        explanation = (
            'This contract has a low model-based risk rating because none of the main proxy indicators are active. '
            'The result should still be interpreted cautiously because the dataset uses proxy labels and incomplete public indicators.'
        )

    report = RiskReport(
        contract_id=str(row.get('contract_id')),
        title=str(row.get('title', ''))[:250],
        buyer_name=None if pd.isna(row.get('buyer_name')) else str(row.get('buyer_name')),
        risk_score=score,
        risk_level=risk_level(score),
        risk_factors=factors,
        retrieved_context=[RetrievedContext(**ctx) for ctx in contexts],
        explanation=explanation,
        human_review_questions=make_review_questions(factors),
        disclaimer='Academic prototype only. This report identifies review priorities, not evidence of fraud or legal wrongdoing.'
    )
    return report

print('Report generator ready.')

## 7. Select Demo Contracts

In [ ]:
N_REPORTS = 12

candidate_df = audit_df.copy()
candidate_df['factor_count'] = candidate_df.apply(lambda row: len(build_risk_factors(row)), axis=1)
candidate_df['prelim_score'] = candidate_df.apply(lambda row: score_risk(build_risk_factors(row), row.get('anomaly_score')), axis=1)

demo_sample = candidate_df.sort_values(
    ['prelim_score', 'factor_count', 'anomaly_score'],
    ascending=[False, False, False]
).head(N_REPORTS).copy()

demo_sample[['contract_id', 'title', 'buyer_name', 'prelim_score', 'factor_count', 'anomaly_score']].head(12)

## 8. Generate Risk Reports

In [ ]:
reports = []
retrieval_records = []

for _, row in demo_sample.iterrows():
    report = explain_contract(row, top_k=3)
    report_dict = report.model_dump() if hasattr(report, 'model_dump') else report.dict()
    reports.append(report_dict)
    for rank, ctx in enumerate(report_dict['retrieved_context'], start=1):
        retrieval_records.append({
            'contract_id': report_dict['contract_id'],
            'rank': rank,
            'source': ctx['source'],
            'topic': ctx['topic'],
            'score': ctx['score'],
            'text': ctx['text'],
        })

with open(OUTPUT_DIR / 'risk_reports_sample.json', 'w', encoding='utf-8') as f:
    json.dump(reports, f, indent=2, ensure_ascii=False)

pd.DataFrame(retrieval_records).to_csv(OUTPUT_DIR / 'retrieved_contexts.csv', index=False)

print(f'Saved {len(reports)} reports to risk_reports_sample.json')
print(f'Saved {len(retrieval_records)} retrieved context rows to retrieved_contexts.csv')

reports[0]

## 9. Visualize Risk Scores

In [ ]:
report_df = pd.DataFrame([
    {
        'contract_id': rpt['contract_id'],
        'title': rpt['title'],
        'risk_score': rpt['risk_score'],
        'risk_level': rpt['risk_level'],
        'factor_count': len(rpt['risk_factors']),
    }
    for rpt in reports
])

colors = report_df['risk_level'].map({'high': '#c0392b', 'medium': '#f39c12', 'low': '#2e86de'}).fillna('#7f8c8d')
labels = report_df['title'].str.slice(0, 55)

plt.figure(figsize=(10, 6))
plt.barh(labels[::-1], report_df['risk_score'][::-1], color=colors[::-1])
plt.axvline(7, linestyle='--', color='#c0392b', label='High risk threshold')
plt.axvline(4, linestyle='--', color='#f39c12', label='Medium risk threshold')
plt.xlabel('Risk score, 0 to 10')
plt.title('RAG Explainability Risk Scores')
plt.xlim(0, 10)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'risk_scores.png', dpi=150)
plt.show()

report_df.to_csv(OUTPUT_DIR / 'risk_report_summary.csv', index=False)
report_df

## 10. Lightweight RAG Evaluation

In [ ]:
def evaluate_report(report):
    factors = set(report['risk_factors'])
    context_text = ' '.join(ctx['text'].lower() + ' ' + ctx['topic'].lower() for ctx in report['retrieved_context'])
    explanation = report['explanation'].lower()

    if not factors:
        context_coverage = 1.0
    else:
        matched = 0
        for factor in factors:
            terms = factor.replace('_', ' ').split()
            if any(term in context_text for term in terms):
                matched += 1
        context_coverage = matched / len(factors)

    schema_complete = all(report.get(key) not in (None, '', []) for key in [
        'contract_id', 'title', 'risk_score', 'risk_level', 'retrieved_context', 'explanation', 'disclaimer'
    ])
    grounded_language = int('not proof' in explanation or 'not evidence' in report['disclaimer'].lower())

    return {
        'contract_id': report['contract_id'],
        'context_coverage': context_coverage,
        'schema_complete': int(schema_complete),
        'grounded_language': grounded_language,
        'retrieved_chunks': len(report['retrieved_context']),
    }

eval_df = pd.DataFrame([evaluate_report(report) for report in reports])
eval_df.to_csv(OUTPUT_DIR / 'rag_evaluation.csv', index=False)

print('Evaluation averages')
print(eval_df[['context_coverage', 'schema_complete', 'grounded_language', 'retrieved_chunks']].mean().round(3))
eval_df

## 11. One-Contract Demo

In [ ]:
demo_report = reports[0]

with open(OUTPUT_DIR / 'risk_report_demo.json', 'w', encoding='utf-8') as f:
    json.dump(demo_report, f, indent=2, ensure_ascii=False)

print('=' * 80)
print('CONTRACT ID:', demo_report['contract_id'])
print('TITLE      :', demo_report['title'])
print('BUYER      :', demo_report.get('buyer_name'))
print('RISK       :', f"{demo_report['risk_score']}/10", demo_report['risk_level'].upper())
print('\nRISK FACTORS')
for factor in demo_report['risk_factors']:
    print('-', factor)

print('\nRETRIEVED CONTEXT')
for ctx in demo_report['retrieved_context']:
    print(f"- {ctx['source']} ({ctx['score']:.3f})")

print('\nEXPLANATION')
print(demo_report['explanation'])

print('\nHUMAN REVIEW QUESTIONS')
for question in demo_report['human_review_questions']:
    print('-', question)

## 12. Deliverables

In [ ]:
deliverables = [
    'rag_knowledge_chunks.csv',
    'risk_reports_sample.json',
    'risk_report_demo.json',
    'retrieved_contexts.csv',
    'risk_report_summary.csv',
    'risk_scores.png',
    'rag_evaluation.csv',
]

print('Notebook 04 deliverables')
print('-' * 70)
for name in deliverables:
    path = OUTPUT_DIR / name
    status = 'OK' if path.exists() else 'MISSING'
    size_kb = path.stat().st_size / 1024 if path.exists() else 0
    print(f'{name:<35} {status:<8} {size_kb:>10.1f} KB')